# CIM Datasets

The purpose of this notebook is to gather metadata for datasets found in [data.colorado.gov](data.colorado.gov), specifically those that have the Colorado Information Marketplace (CIM) listed as the dataset owner.

The SoDapy python package was used to retrieve the publicly available datasets, since it contains a `Socrata` object to use as a client for requests. This python package can be installed using

`pip install sodapy`

more information along with some examples can be found in
* [GitHub repo](https://github.com/xmunoz/sodapy#datasetslimit0-offset0) (archived)
* [API Docs](https://dev.socrata.com/consumers/getting-started.html)

Another link with a useful example for getting started was found [here](https://holowczak.com/getting-started-with-nyc-opendata-and-the-socrata-api/5/)

Required packages:


In [2]:
! pip install sodapy

In [1]:
from sodapy import Socrata
import pandas as pd

Unfortunately most of the API calls that could be made with this client required a dataset id to be passed in as a paramater. Therefore all of the datasets found in the data catalog were pulled in and filtered using conditionals with pandas.

In [2]:
cim_url_query = 'data.colorado.gov'
datasets = None

with Socrata(cim_url_query, None) as client:
    datasets = client.datasets()

Some discovery work showed that there wasn't an owner name found within each JSON, but there did exist an owner id. Since this notebook was intended to filter out any datasets not relating to CIM, it was safe to hard code the CIM name as the dataset owner. Another thing to note was that when the dataset JSON was converted to a pandas data frame, each row had a corresponding name instead of a numerical index. To locate the row of interest, the `.loc[[]]` syntax was used. After locating the necessary information found in the JSON, the row was appended to the result data frame.

In [5]:
res = pd.DataFrame()
cim_dataset_owner_id = '8cet-tw9x'
for dset in datasets:
    dset_df = pd.DataFrame(dset)
    if (dset_df.loc[['type']]['resource'].item() == 'dataset'
        and dset_df.loc[['id']]['owner'].item() == cim_dataset_owner_id):
        page_views = dset_df.loc[['page_views']]['resource'].item()
        row = {
            "name": dset_df.loc[['name']]['resource'].item(),
            "id": dset_df.loc[['id']]['resource'].item(),
            "link": dset_df.loc[['id']]['link'].item(),
            "owner": 'Colorado Information Marketplace',
            "owner_id": dset_df.loc[['id']]['owner'].item(),
            "attribution": dset_df.loc[['attribution']]['resource'].item(),
            "attribution_link": dset_df.loc[['attribution_link']]['resource'].item(),
            "createdAt": dset_df.loc[['createdAt']]['resource'].item(),
            "data_updated_at": dset_df.loc[['data_updated_at']]['resource'].item(),
            "metadata_updated_at": dset_df.loc[['metadata_updated_at']]['resource'].item(),
            "publication_date": dset_df.loc[['publication_date']]['resource'].item(),
            "page_views_total": page_views['page_views_total'],
            "page_views_total_log": page_views['page_views_total_log'],
            "page_views_last_week": page_views['page_views_last_week'],
            "page_views_last_week_log": page_views['page_views_last_week_log'],
            "page_views_last_month": page_views['page_views_last_month'],
            "page_views_last_month_log": page_views['page_views_last_month_log'],
            "download_count": dset_df.loc[['download_count']]['resource'].item()
        }
        res = pd.concat([res, pd.DataFrame([row])], ignore_index=True)

The results were written out to csv with index turned off to avoid a duplicate index column.

In [6]:
res.head()

,name,id,link,owner,owner_id,attribution,attribution_link,createdAt,data_updated_at,metadata_updated_at,publication_date,page_views_total,page_views_total_log,page_views_last_week,page_views_last_week_log,page_views_last_month,page_views_last_month_log,download_count
0,Business Entities in Colorado,4ykn-tg5h,https://data.colorado.gov/Business/Business-En...,Colorado Information Marketplace,8cet-tw9x,CDOS,https://www.sos.state.co.us/,2014-03-19T22:33:57.000Z,2024-01-05T12:23:54.000Z,2024-01-05T11:59:32.000Z,2018-03-07T16:30:41.000Z,98795,16.592165,234,7.876517,1052,10.040290,34017
1,Professional and Occupational Licenses in Colo...,7s5z-vewr,https://data.colorado.gov/Regulations/Professi...,Colorado Information Marketplace,8cet-tw9x,DORA,https://www.colorado.gov/dora,2016-03-31T21:59:36.000Z,2024-01-05T10:31:27.000Z,2024-01-05T12:01:59.000Z,2018-06-07T19:20:49.000Z,25516,14.639171,143,7.169925,699,9.451211,4451
2,Population Projections in Colorado,q5vp-adf3,https://data.colorado.gov/Demographics/Populat...,Colorado Information Marketplace,8cet-tw9x,DOLA,https://www.colorado.gov/dola,2014-01-28T23:36:26.000Z,2023-11-27T18:06:55.000Z,2024-01-05T11:59:44.000Z,2021-01-12T22:35:05.000Z,23061,14.493230,106,6.741467,325,8.348728,3434
3,Uniform Commercial Code (UCC) Filing Informati...,wffy-3uut,https://data.colorado.gov/Business/Uniform-Com...,Colorado Information Marketplace,8cet-tw9x,CDOS,https://www.sos.state.co.us/,2014-03-09T07:33:16.000Z,2024-01-05T10:05:10.000Z,2024-01-05T11:59:27.000Z,2016-11-22T23:54:53.000Z,12175,13.571753,38,5.285402,169,7.409391,3880
4,Trade Names for Businesses in Colorado,u7sb-g482,https://data.colorado.gov/Business/Trade-Names...,Colorado Information Marketplace,8cet-tw9x,CDOS,https://www.sos.state.co.us/,2014-01-08T19:34:11.000Z,2024-01-04T18:40:54.000Z,2024-01-05T12:02:47.000Z,2022-06-08T04:27:33.000Z,11673,13.511011,36,5.209453,151,7.247928,13831


In [7]:
res.to_csv('cim_datasets.csv', index=False)